In [1]:
from pathlib import Path
corpus_dir = Path("../10k")

html_files = sorted(corpus_dir.glob("*.html"))
len(html_files),html_files[:5]


(497,
 [PosixPath('../10k/A.html'),
  PosixPath('../10k/AAL.html'),
  PosixPath('../10k/AAPL.html'),
  PosixPath('../10k/ABBV.html'),
  PosixPath('../10k/ABNB.html')])

In [2]:
sample_tickers = ["ADI", "AAPL", "CBOE", "MSFT", "NVDA"]
sample_files = [corpus_dir/f"{ticker}.html" for ticker in sample_tickers]
[(path.name, path.exists()) for path in sample_files]


[('ADI.html', True),
 ('AAPL.html', True),
 ('CBOE.html', True),
 ('MSFT.html', True),
 ('NVDA.html', True)]

In [3]:
import sec_parser as sp

from sec_parser.semantic_elements import (
    TextElement,
    TitleElement,
    TableElement,
    TopSectionTitle,
    SupplementaryText,
)

parser = sp.Edgar10KParser()

In [4]:
def parse_10k_file(path):
    html = path.read_text(encoding="utf-8", errors='ignore')
    elements = parser.parse(html)

    chunks = []
    section_title = None
    ticker = path.stem

    for order, element in enumerate(elements):
        if isinstance(element, (TopSectionTitle, TitleElement)):
            section_title = element.text.strip()
            continue

        elif isinstance(element, TextElement):
            text = element.text.strip()
            chunk_type = 'text'
        elif isinstance(element, TableElement):
            text = element.table_to_markdown().strip()
            chunk_type = 'table'

        elif isinstance(element, SupplementaryText):
            text = element.text.strip()
            chunk_type = 'supplementary_text'
        
        else: 
            continue
        
        if not text:
            continue
        
        chunks.append({
            "ticker": ticker,
            "order": order,
            "chunk_type": chunk_type,
            "section_title": section_title,
            "text": text,
        })

    return chunks



In [5]:
sample_chunks = []

for path in sample_files:
    file_chunks = parse_10k_file(path)
    sample_chunks.extend(file_chunks)

len(sample_chunks)

1878

In [6]:
import pandas as pd
sample_corpus_df = pd.DataFrame(sample_chunks)

print(sample_corpus_df.shape)
sample_corpus_df.head()

(1878, 5)


,ticker,order,chunk_type,section_title,text
0,ADI,16,text,"Company Overview, Strategy and Mission","Analog Devices, Inc. (we, Analog Devices or th..."
1,ADI,18,text,Available Information,We maintain a website with the address www.ana...
2,ADI,20,text,Products,Semiconductor components are the building bloc...
3,ADI,22,text,Sales Channel,We sell our products globally through a direct...
4,ADI,24,text,Markets,The breakdown of our annual revenue by end mar...


In [7]:
sample_corpus_df["ticker"].value_counts()

ticker
CBOE    477
MSFT    410
ADI     389
NVDA    308
AAPL    294
Name: count, dtype: int64

In [8]:
sample_corpus_df["chunk_type"].value_counts()

chunk_type
text                  1338
table                  374
supplementary_text     166
Name: count, dtype: int64

In [9]:
sample_corpus_df.groupby(["ticker", "chunk_type"]).size()

ticker  chunk_type        
AAPL    supplementary_text     43
        table                  48
        text                  203
ADI     supplementary_text     46
        table                  87
        text                  256
CBOE    supplementary_text     32
        table                  94
        text                  351
MSFT    supplementary_text     27
        table                  86
        text                  297
NVDA    supplementary_text     18
        table                  59
        text                  231
dtype: int64

In [11]:
sample_corpus_df["text_len"] = sample_corpus_df["text"].str.len()

sample_corpus_df["text_len"].describe()

count     1878.000000
mean      1059.402556
std       1829.938520
min          3.000000
25%        202.000000
50%        531.000000
75%       1271.750000
max      44628.000000
Name: text_len, dtype: float64

In [12]:
sample_corpus_df.sort_values("text_len", ascending=False)[
    ["ticker", "order", "chunk_type", "section_title", "text_len"]
].head(10)

,ticker,order,chunk_type,section_title,text_len
764,CBOE,371,text,Risks Relating to Our Business,44628
701,CBOE,236,text,Competition,20231
1616,NVDA,107,text,Our operations could be affected by the comple...,19241
705,CBOE,245,text,EU Transparency Rules,15360
1603,NVDA,79,text,Failure to estimate customer demand accurately...,13483
774,CBOE,393,text,Risks Relating to Our Cboe Digital Business,12482
745,CBOE,333,text,Risks Relating to Our Business,11733
896,CBOE,761,text,Commercial Commitments and Contractual Obligat...,11241
713,CBOE,264,text,Regulatory Services Agreement with OCC,11076
698,CBOE,228,text,U.S. Tape Plans,10792


In [13]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

embedding_model_name = "BAAI/bge-small-en-v1.5"

tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=400,
    chunk_overlap=50,
)

/home/wagyu0923/miniconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"])

In [15]:
longest_row = sample_corpus_df.sort_values(
    "text_len",
    ascending=False,).iloc[0]

longest_row

ticker                                                        CBOE
order                                                          371
chunk_type                                                    text
section_title                       Risks Relating to Our Business
text             Table of Contentssettlement on all of our matc...
text_len                                                     44628
Name: 764, dtype: object

In [18]:
count_tokens(longest_row["text"])

8387

In [23]:
def build_embedding_text(section_title, text):
    return f"{section_title}\n\n{text}" if section_title else text

def split_row(row):
    parts = text_splitter.split_text(row["text"])

    records=[]

    for part in parts:
        records.append({
            "ticker": row["ticker"],
            "chunk_type": row["chunk_type"],
            "section_title": row["section_title"],
            "text": part,
            "embedding_text": build_embedding_text(row["section_title"],part)
        })

    return records


In [24]:
final_rows = []

for _, row in sample_corpus_df.iterrows():
    final_rows.extend(split_row(row))

sample_final_df = pd.DataFrame(final_rows)

sample_final_df = sample_final_df.reset_index(drop=True)
sample_final_df["chunk_id"] = sample_final_df.index
sample_final_df = sample_final_df[
    ["chunk_id", "ticker", "chunk_type", "section_title", "text", "embedding_text"]
]

sample_final_df


,chunk_id,ticker,chunk_type,section_title,text,embedding_text
0,0,ADI,text,"Company Overview, Strategy and Mission","Analog Devices, Inc. (we, Analog Devices or th...","Company Overview, Strategy and Mission\n\nAnal..."
1,1,ADI,text,"Company Overview, Strategy and Mission",from our acquisitions to complement our R&D an...,"Company Overview, Strategy and Mission\n\nfrom..."
2,2,ADI,text,"Company Overview, Strategy and Mission",is listed on the Nasdaq Global Select Market u...,"Company Overview, Strategy and Mission\n\nis l..."
3,3,ADI,text,Available Information,We maintain a website with the address www.ana...,Available Information\n\nWe maintain a website...
4,4,ADI,text,Products,Semiconductor components are the building bloc...,Products\n\nSemiconductor components are the b...
...,...,...,...,...,...,...
2438,2438,NVDA,text,Signatures,Pursuant to the requirements of Section 13 or ...,Signatures\n\nPursuant to the requirements of ...
2439,2439,NVDA,table,Signatures,| | | | | | NVIDIA Corporation |\n| | | By: | ...,Signatures\n\n| | | | | | NVIDIA Corporation |...
2440,2440,NVDA,text,Power of Attorney,"KNOW ALL PERSONS BY THESE PRESENTS, that each ...",Power of Attorney\n\nKNOW ALL PERSONS BY THESE...
2441,2441,NVDA,table,Power of Attorney,| | | Signature | | | Title | | | Date |\n| | ...,Power of Attorney\n\n| | | Signature | | | Tit...
